# K-Means Clustering — Hands-on Lab
**HBE_2206 | Hazem Elshafey**

This notebook follows the K-Means reference notes and adds practical experiments with scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42

## 1. Create a small dataset

We generate three compact groups so the mechanics of K-Means are easy to inspect. This is synthetic data, not a real customer dataset.

In [ ]:
X, y_reference = make_blobs(n_samples=450, centers=3, cluster_std=0.75, random_state=RANDOM_STATE)
plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], s=25)
plt.title('Synthetic data before clustering')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

## 2. Fit K-Means

The generated reference labels are used only to create the visual example; K-Means itself receives X only and does not use those labels.

In [ ]:
kmeans = KMeans(n_clusters=3, init='k-means++', n_init=10, max_iter=300, random_state=RANDOM_STATE)
labels = kmeans.fit_predict(X)
centers = kmeans.cluster_centers_
print('Inertia:', round(kmeans.inertia_, 2))
print('Iterations:', kmeans.n_iter_)
print('Cluster sizes:')
print(pd.Series(labels).value_counts().sort_index())

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=labels, s=25)
plt.scatter(centers[:, 0], centers[:, 1], marker='X', s=220)
plt.title('K-Means assignments and centroids')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

## 3. Compare candidate values of K

Inertia always tends to decrease as K increases. Look for diminishing returns, then combine the elbow plot with silhouette score and domain context.

In [ ]:
records = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    candidate_labels = model.fit_predict(X)
    records.append({'K': k, 'inertia': model.inertia_, 'silhouette': silhouette_score(X, candidate_labels)})

scores = pd.DataFrame(records)
scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(scores['K'], scores['inertia'], marker='o')
axes[0].set(title='Elbow method', xlabel='K', ylabel='Inertia')
axes[1].plot(scores['K'], scores['silhouette'], marker='o')
axes[1].set(title='Silhouette score', xlabel='K', ylabel='Score')
plt.tight_layout()
plt.show()

## 4. Why feature scaling matters

Create a dataset where the second feature has a much larger numeric scale. Compare the raw distances with standardized features.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
small_scale = rng.normal(loc=0, scale=1, size=(250, 1))
large_scale = rng.normal(loc=0, scale=100, size=(250, 1))
X_mixed = np.hstack([small_scale, large_scale])
X_scaled = StandardScaler().fit_transform(X_mixed)

raw_model = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X_mixed)
scaled_model = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
print('Raw feature standard deviations:', X_mixed.std(axis=0))
print('Scaled feature standard deviations:', X_scaled.std(axis=0))
print('Raw-space inertia:', round(raw_model.inertia_, 2))
print('Scaled-space inertia:', round(scaled_model.inertia_, 2))
print('Note: inertia values from different feature spaces are not directly comparable.')

## 5. Practice

1. Change K from 3 to 2 and 4. Describe what changes in the plot.
2. Change `random_state`. Do the centroids move?
3. Increase `cluster_std` in `make_blobs`. How does cluster overlap affect silhouette?
4. Explain why the generated reference labels were not passed to `fit_predict`.
5. Write one limitation of K-Means for this dataset and one for a non-spherical dataset.

## Key reminders
- K-Means is unsupervised: it does not need class labels.
- It minimizes within-cluster squared distances.
- Scaling matters when feature units/ranges differ.
- Inertia alone cannot select K.
- Cluster usefulness must be interpreted in context.

Run all cells from top to bottom in a fresh Colab runtime before using the notebook as a validated teaching resource.